Este script implementa paso a paso los cálculos descritos en las Tablas 1.2, 1.4, 1.5, 1.6 y 1.7,
usando estimaciones bayesianas, distribuciones binomial y de Poisson, y convolución entre ellas.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import betabinom, binom, poisson

In [ ]:
# 1. Datos iniciales (tabla 1.1)
grades = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "CC", "C"]
num_obligors = [34, 56, 119, 257, 191, 102, 50, 34, 12]
num_defaults = [1, 1, 3, 2, 2, 6, 3, 1, 2]
total_defaults = sum(num_defaults)

In [ ]:
# 2. Cálculo de estimaciones bayesianas con priors no informativos (Tabla 1.2)
bayesian_estimates = [(d / total_defaults) * 100 for d in num_defaults]
df_bayesian = pd.DataFrame({
    "Grade": grades,
    "Obligors": num_obligors,
    "Defaults": num_defaults,
    "Bayesian Estimates (%)": bayesian_estimates
})
print("=== Tabla 1.2 - Estimaciones Bayesianas ===")
print(df_bayesian, "\n")

=== Tabla 1.2 - Estimaciones Bayesianas ===
  Grade  Obligors  Defaults  Bayesian Estimates (%)
0   AAA        34         1                4.761905
1    AA        56         1                4.761905
2     A       119         3               14.285714
3   BBB       257         2                9.523810
4    BB       191         2                9.523810
5     B       102         6               28.571429
6   CCC        50         3               14.285714
7    CC        34         1                4.761905
8     C        12         2                9.523810 



In [ ]:
# 3. Distribución binomial para categoría BBB (Tabla 1.4)
p_bbb = bayesian_estimates[3] / 100
x_binomial = np.arange(0, 21)
binomial_bbb = binom.pmf(x_binomial, total_defaults, p_bbb)
df_binomial_bbb = pd.DataFrame({
    "x": x_binomial,
    "P(X=x)": binomial_bbb
})
print("=== Tabla 1.4 - Binomial para BBB ===")
print(df_binomial_bbb.head(10), "\n")

=== Tabla 1.4 - Binomial para BBB ===
   x    P(X=x)
0  0  0.122242
1  1  0.270219
2  2  0.284441
3  3  0.189627
4  4  0.089824
5  5  0.032147
6  6  0.009024
7  7  0.002035
8  8  0.000375
9  9  0.000057 



In [ ]:
# 4. Cálculo de lambda (frecuencia de defaults) por grado (Tabla 1.5)
lambda_values = [d / o if o > 0 else 0 for d, o in zip(num_defaults, num_obligors)]
df_lambda = pd.DataFrame({
    "Grade": grades,
    "Obligors": num_obligors,
    "Defaults": num_defaults,
    "Lambda": lambda_values
})
print("=== Tabla 1.5 - Lambdas ===")
print(df_lambda, "\n")

=== Tabla 1.5 - Lambdas ===
  Grade  Obligors  Defaults    Lambda
0   AAA        34         1  0.029412
1    AA        56         1  0.017857
2     A       119         3  0.025210
3   BBB       257         2  0.007782
4    BB       191         2  0.010471
5     B       102         6  0.058824
6   CCC        50         3  0.060000
7    CC        34         1  0.029412
8     C        12         2  0.166667 



In [ ]:
# 5. Distribución de Poisson para BBB (Tabla 1.6)
lambda_bbb = lambda_values[3]
n_poisson = np.arange(0, 8)
poisson_bbb = poisson.pmf(n_poisson, lambda_bbb)
df_poisson_bbb = pd.DataFrame({
    "n": n_poisson,
    "P(N=n)": poisson_bbb
})
print("=== Tabla 1.6 - Poisson para BBB ===")
print(df_poisson_bbb, "\n")

=== Tabla 1.6 - Poisson para BBB ===
   n        P(N=n)
0  0  9.922481e-01
1  1  7.721775e-03
2  2  3.004582e-05
3  3  7.793986e-08
4  4  1.516340e-10
5  5  2.360062e-13
6  6  3.061040e-16
7  7  3.403046e-19 



In [ ]:
# 6. Convolución Binomial x Poisson para BBB (inicio de metodología Tabla 1.7)
convoluted_bbb = np.convolve(binomial_bbb, poisson_bbb, mode='full')[:len(binomial_bbb)]
df_convolution_bbb = pd.DataFrame({
    "x": x_binomial,
    "Convoluted P(X=x)": convoluted_bbb
})

In [ ]:
# 7. Repetir convolución para todas las calificaciones y obtener PDs implícitas
pd_implied = []
for i in range(len(grades)):
    p_i = bayesian_estimates[i] / 100
    lambda_i = lambda_values[i]
    d_i = num_defaults[i]

    binomial_i = binom.pmf(np.arange(0, total_defaults + 1), total_defaults, p_i)
    poisson_i = poisson.pmf(np.arange(0, 8), lambda_i)
    conv_i = np.convolve(binomial_i, poisson_i, mode='full')[:len(binomial_i)]
    pd_val = conv_i[d_i]  if d_i < len(conv_i) else 0
    pd_implied.append(pd_val)

In [ ]:
# 8. Comparación con los valores reportados en la Tabla 1.7
pd_article = [.0108, .0174, .0233, .0255, .0285, .0390, .0528, .0636, .1046]
df_comparison = pd.DataFrame({
    "Grade": grades,
    "PD Article": pd_article,
    "PD Calculated": pd_implied,
    "Difference": [abs(a - c) for a, c in zip(pd_article, pd_implied)]
})
print("=== Tabla 1.7 - Comparación Final ===")
print(df_comparison)

=== Tabla 1.7 - Comparación Final ===
  Grade  PD Article  PD Calculated  Difference
0   AAA      0.0108       0.376217    0.365417
1    AA      0.0174       0.376515    0.359115
2     A      0.0233       0.241487    0.218187
3   BBB      0.0255       0.284326    0.258826
4    BB      0.0285       0.284285    0.255785
5     B      0.0390       0.188993    0.149993
6   CCC      0.0528       0.240930    0.188130
7    CC      0.0636       0.376217    0.312617
8     C      0.1046       0.280334    0.175734


Modelo de PD Implícita mediante Esperanza de la Convolución Beta-Binomial × Poisson con prior de Jeffreys

In [ ]:
# Parámetros bayesianos (Prior de Jeffreys)
alpha_prior = 0.5
beta_prior = 0.5
max_defaults = 10
max_poisson = 7

pd_esperada = []

for i in range(len(grades)):
    n = num_obligors[i]
    x = num_defaults[i]
    lambda_i = x / n if n > 0 else 0

    a_post = alpha_prior + x
    b_post = beta_prior + n - x

    beta_binom_probs = [betabinom.pmf(k, n, a_post, b_post) for k in range(max_defaults + 1)]
    poisson_probs = [poisson.pmf(k, lambda_i) for k in range(max_poisson + 1)]
    conv = np.convolve(beta_binom_probs, poisson_probs)

    esperanza_defaults = np.sum(np.arange(len(conv)) * conv)
    pd_val = (esperanza_defaults / n) * 100
    pd_esperada.append(pd_val)

# Tabla comparativa final
df_result = pd.DataFrame({
    "Grades": grades,
    "PD del Artículo (%)": pd_article,
    "PD Esperada Convolución (%)": pd_esperada,
    "Diferencia Absoluta (%)": [round(abs(a - b), 4) for a, b in zip(pd_article, pd_esperada)]
})

print("=== Comparativa PD Artículo vs Modelo con Datos Confirmados ===")
print(df_result)


=== Comparativa PD Artículo vs Modelo con Datos Confirmados ===
  Grades  PD del Artículo (%)  PD Esperada Convolución (%)  \
0    AAA               0.0108                     4.352421   
1     AA               0.0174                     2.645570   
2      A               0.0233                     2.772650   
3    BBB               0.0255                     0.944564   
4     BB               0.0285                     1.271741   
5      B               0.0390                     4.798754   
6    CCC               0.0528                     6.675702   
7     CC               0.0636                     4.352421   
8      C               0.1046                    20.606651   

   Diferencia Absoluta (%)  
0                   4.3416  
1                   2.6282  
2                   2.7493  
3                   0.9191  
4                   1.2432  
5                   4.7598  
6                   6.6229  
7                   4.2888  
8                  20.5021  


Ajuste Exponencial de Probabilidades de Default (PD) sobre valores esperados


In [ ]:
from scipy.optimize import curve_fit
# Datos base
indices_grades = np.arange(1, 10)

# Función exponencial
def exponential(x, a, b):
    return a * np.exp(b * x)

# Ajuste de curva
params, _ = curve_fit(exponential, indices_grades, pd_esperada, p0=(0.5, 0.1))
a_fit, b_fit = params
pd_ajustada = exponential(indices_grades, a_fit, b_fit)

# Resultado en DataFrame
df_ajuste = pd.DataFrame({
    "Grades": grades,
    "PD Esperada (%)": pd_esperada,
    "PD Ajustada Exponencial (%)": pd_ajustada,
    "Diferencia (%)": np.round(np.abs(np.array(pd_esperada) - pd_ajustada), 4)
})

print("=== Ajuste Exponencial de PDs ===")
print(df_ajuste)

=== Ajuste Exponencial de PDs ===
  Grades  PD Esperada (%)  PD Ajustada Exponencial (%)  Diferencia (%)
0    AAA         4.352421                     0.064116          4.2883
1     AA         2.645570                     0.130639          2.5149
2      A         2.772650                     0.266183          2.5065
3    BBB         0.944564                     0.542359          0.4022
4     BB         1.271741                     1.105081          0.1667
5      B         4.798754                     2.251650          2.5471
6    CCC         6.675702                     4.587835          2.0879
7     CC         4.352421                     9.347914          4.9955
8      C        20.606651                    19.046781          1.5599
